# Heritage Short: score candidates with ACE-Step 1.5

Settings: **Accelerator = GPU T4**, **Internet = on**. Run all cells.

Before running, replace the values in the config cell with the block printed by
`python run.py --stage music-plan` (run it after the narration exists).
Download `music_*.wav` and `runs_kaggle.jsonl` into the project's `outputs/music/`.

In [ ]:
# Config: pasted from storyboard.json -> music
CAPTION = 'gentle Ethiopian-inspired instrumental, soft plucked krar-like strings, warm pad, sparse, nostalgic, slow build, motif returns at the end, no vocals'
BPM = 72
KEY = 'D minor'
DURATION_S = 87.875
SEEDS = [11, 12, 13]
INSTRUMENTAL = True

# Models: turbo DiT, and the 1.7B LM that docs/en/GPU_COMPATIBILITY.md recommends for 12-20 GB VRAM
DIT_MODEL = "acestep-v15-turbo"
LM_MODEL = "acestep-5Hz-lm-1.7B"

REPO = "/kaggle/temp/ACE-Step-1.5"
OUT = "/kaggle/working"

## 1. Install uv, clone ACE-Step 1.5, sync its environment

In [ ]:
import os, subprocess

# Kaggle's notebook plotting backend does not exist inside the new environment.
os.environ["MPLBACKEND"] = "Agg"

def sh(*args, cwd=None):
    print("$", " ".join(args))
    subprocess.run(args, cwd=cwd, check=True)

sh("pip", "install", "-q", "uv")
if not os.path.isdir(REPO):
    sh("git", "clone", "--depth", "1", "https://github.com/ACE-Step/ACE-Step-1.5", REPO)
sh("uv", "sync", "--python", "3.11", cwd=REPO)  # pyproject.toml requires Python 3.11-3.12
sh("git", "rev-parse", "HEAD", cwd=REPO)
sh("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv")

## 2. Read the documented interface

The generation script uses only `docs/en/INFERENCE.md`: *Quick Start > Basic Usage*,
*GenerationParams Parameters*, *GenerationConfig Parameters* and *Best Practices > 7. Memory Management*.
This cell prints those sections from the cloned repo, then checks every parameter name we use
against the installed classes. It stops if the repo has changed.

In [ ]:
import re
from pathlib import Path

doc = Path(REPO, "docs/en/INFERENCE.md").read_text(encoding="utf-8")

def section(title):
    match = re.search(rf"^(#+) {re.escape(title)}\n", doc, flags=re.M)
    if not match:
        raise RuntimeError(f"docs/en/INFERENCE.md no longer has the section: {title}")
    level = len(match.group(1))
    rest = doc[match.end():]
    # Stop at the next heading of the same or a higher level, ignoring comments in code blocks.
    lines, in_code = [], False
    for line in rest.splitlines():
        if line.startswith("```"):
            in_code = not in_code
        heading = re.match(r"^(#+) ", line)
        if heading and not in_code and len(heading.group(1)) <= level:
            break
        lines.append(line)
    return f"{match.group(0)}" + "\n".join(lines)

for title in ("Basic Usage", "GenerationParams Parameters", "GenerationConfig Parameters",
              "7. Memory Management"):
    print("=" * 30, "docs/en/INFERENCE.md >", title, "=" * 30)
    print(section(title))

check = """
from dataclasses import fields
from acestep.inference import GenerationParams, GenerationConfig
need_params = {"task_type", "caption", "lyrics", "instrumental", "bpm", "keyscale", "duration",
               "inference_steps", "shift", "seed", "thinking", "use_cot_metas", "use_cot_caption",
               "use_cot_language"}
need_config = {"batch_size", "use_random_seed", "seeds", "audio_format"}
missing = (need_params - {f.name for f in fields(GenerationParams)}) | (
    need_config - {f.name for f in fields(GenerationConfig)})
assert not missing, f"Parameters not found in this version: {missing}"
print("All parameter names exist.")
"""
sh("uv", "run", "python", "-c", check, cwd=REPO)

## 3 and 4. Generate three instrumental candidates and log each one

In [ ]:
import json

settings = {"caption": CAPTION, "bpm": BPM, "key": KEY, "duration_s": DURATION_S, "seeds": SEEDS,
            "instrumental": INSTRUMENTAL, "dit_model": DIT_MODEL, "lm_model": LM_MODEL,
            "repo": REPO, "out": OUT}
Path(OUT, "music_settings.json").write_text(json.dumps(settings), encoding="utf-8")

In [ ]:
%%writefile /kaggle/working/generate_music_candidates.py
"""Runs inside ACE-Step's uv environment. Interface: docs/en/INFERENCE.md."""
from datetime import datetime, timezone
import json
from pathlib import Path
import shutil
import time

import torch

from acestep.handler import AceStepHandler
from acestep.inference import GenerationConfig, GenerationParams, generate_music
from acestep.llm_inference import LLMHandler

s = json.loads(Path("/kaggle/working/music_settings.json").read_text(encoding="utf-8"))
out = Path(s["out"])
log_path = out / "runs_kaggle.jsonl"
gpu = torch.cuda.get_device_name(0)

dit = AceStepHandler()
status, ok = dit.initialize_service(
    project_root=s["repo"], config_path=s["dit_model"], device="cuda",
    offload_to_cpu=True, quantization="int8_weight_only", compile_model=True,
)
print(status)
assert ok, "DiT initialisation failed"

llm = LLMHandler()
for backend in ("vllm", "pt"):
    status, ok = llm.initialize(
        checkpoint_dir=str(Path(s["repo"], "checkpoints")), lm_model_path=s["lm_model"],
        backend=backend, device="cuda",
    )
    print(backend, status)
    if ok:
        break
assert ok, "LM initialisation failed"

for seed in s["seeds"]:
    params = GenerationParams(
        task_type="text2music",
        caption=s["caption"],
        lyrics="[Instrumental]",
        instrumental=s["instrumental"],
        bpm=s["bpm"],
        keyscale=s["key"],
        duration=s["duration_s"],
        inference_steps=8,
        shift=3.0,  # recommended for turbo models
        seed=seed,
        thinking=True,
        # Keep our caption and metadata exactly as written in the storyboard.
        use_cot_metas=False,
        use_cot_caption=False,
        use_cot_language=False,
    )
    config = GenerationConfig(batch_size=1, use_random_seed=False, seeds=[seed], audio_format="wav")
    started = time.monotonic()
    error = None
    try:
        result = generate_music(dit, llm, params, config, save_dir=str(out / "raw"))
        if result.success:
            shutil.copyfile(result.audios[0]["path"], out / f"music_{seed}.wav")
        else:
            error = str(result.error)
    except Exception as exc:  # one failed seed must not stop the others
        error = f"{type(exc).__name__}: {exc}"
    latency = time.monotonic() - started

    entry = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "stage": "music",
        "shot_id": f"music_{seed}",
        "provider": "kaggle-t4",
        "model": f"ACE-Step-1.5 {s['dit_model']} + {s['lm_model']}",
        "params": {"caption": s["caption"], "bpm": s["bpm"], "key": s["key"],
                   "duration_s": s["duration_s"], "instrumental": s["instrumental"],
                   "inference_steps": 8, "shift": 3.0, "gpu": gpu,
                   "gpu_minutes": round(latency / 60, 3)},
        "seed": seed,
        "latency_s": round(latency, 4),
        "cost_usd_equiv": 0.0,
        "cache_hit": False,
        "status": "ok" if error is None else "error",
        "error": error,
    }
    with log_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(seed, entry["status"], f"{latency:.1f}s", error or "")

In [ ]:
sh("uv", "run", "python", "/kaggle/working/generate_music_candidates.py", cwd=REPO)

In [ ]:
from IPython.display import Audio, display

print(Path(OUT, "runs_kaggle.jsonl").read_text(encoding="utf-8"))
for seed in SEEDS:
    path = Path(OUT, f"music_{seed}.wav")
    if path.is_file():
        print(path.name)
        display(Audio(str(path)))